In [2]:
import numpy as np

class HMMViterbi:
    def __init__(self, states, observations, start_prob, trans_prob, emit_prob):
        self.states = states
        self.observations = observations
        self.start_prob = start_prob
        self.trans_prob = trans_prob
        self.emit_prob = emit_prob

    def run_viterbi(self, obs_sequence):
        """
        Implementation of the Viterbi Algorithm to find the most likely hidden state sequence.
        """
        T = len(obs_sequence)
        N = len(self.states)

        # v_trellis[state, time] stores the max probability to reach that state at that time
        v_trellis = np.zeros((N, T))
        # path_ptr stores the index of the state that led to the max probability
        path_ptr = np.zeros((N, T), dtype=int)

        # initialize Step
        first_obs_idx = self.observations.index(obs_sequence[0])
        for s in range(N):
            v_trellis[s, 0] = self.start_prob[self.states[s]] * self.emit_prob[self.states[s]][obs_sequence[0]]

        # recursion step
        for t in range(1, T):
            for s in range(N):
                # calculate probabilities of coming from every possible previous state
                probs = [v_trellis[prev_s, t-1] * self.trans_prob[self.states[prev_s]][self.states[s]] * self.emit_prob[self.states[s]][obs_sequence[t]]
                         for prev_s in range(N)]

                v_trellis[s, t] = max(probs)
                path_ptr[s, t] = np.argmax(probs)

        # path backtracking
        best_path = np.zeros(T, dtype=int)
        best_path[T-1] = np.argmax(v_trellis[:, T-1])

        for t in range(T-2, -1, -1):
            best_path[t] = path_ptr[best_path[t+1], t+1]

        # convert indices back to state names
        return [self.states[i] for i in best_path], np.max(v_trellis[:, T-1])

# example
if __name__ == "__main__":
    # define the Model
    states = ('Rainy', 'Sunny')
    observations = ('walk', 'shop', 'clean')

    start_p = {'Rainy': 0.6, 'Sunny': 0.4}

    trans_p = {
        'Rainy': {'Rainy': 0.7, 'Sunny': 0.3},
        'Sunny': {'Rainy': 0.4, 'Sunny': 0.6},
    }

    emit_p = {
        'Rainy': {'walk': 0.1, 'shop': 0.4, 'clean': 0.5},
        'Sunny': {'walk': 0.6, 'shop': 0.3, 'clean': 0.1},
    }

    model = HMMViterbi(states, observations, start_p, trans_p, emit_p)

    # input sequence of activities
    test_obs = ('walk', 'shop', 'clean')

    path, prob = model.run_viterbi(test_obs)

    print(f"Sequence of Activities: {test_obs}")
    print(f"Most Likely Weather Sequence: {path}")
    print(f"Probability of this path: {prob:.6f}")

Sequence of Activities: ('walk', 'shop', 'clean')
Most Likely Weather Sequence: ['Sunny', 'Rainy', 'Rainy']
Probability of this path: 0.013440
